# Starter Notebook: Prediksi Biaya Operasional Cabang

Bootcamp Data Analytics, AI, dan Big Data · Officer Development Program BNI · Pusilkom UI

Notebook ini membawa Anda dari nol sampai **submission pertama yang valid** dalam waktu sekitar 10 menit. Setelah itu, tugas Anda adalah mengalahkannya.

Seluruh teknik yang dibutuhkan sudah dibahas pada Hands-on Lab Regresi. Kalau lupa, buka lagi notebook lab tersebut.

## Aturan Main Singkat

- Prediksi kolom `BiayaOperasional_jutaPerBulan` untuk setiap `IDCabang` di `test.csv`.
- Metrik penilaian **RMSE**, semakin kecil semakin baik.
- Leaderboard yang terlihat selama lomba hanya memakai 30 persen data uji. Peringkat akhir memakai 70 persen sisanya. Jadi jangan mengejar skor publik secara berlebihan.

## 1. Persiapan

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, OneHotEncoder
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
TARGET = 'BiayaOperasional_jutaPerBulan'
ID = 'IDCabang'

print('Library siap.')

### Memuat Data

Cell berikut mencari data secara otomatis, jadi notebook ini **jalan di mana saja tanpa Anda ubah sedikit pun**.

- Di **Kaggle Notebook**, data diambil dari folder input kompetisi.
- Di **Google Colab atau laptop sendiri**, data ditulis dari salinan yang sudah ditanam di dalam notebook ini. Anda tidak perlu mengunggah apa pun.

Isi cell ini tidak perlu dipahami. Cukup jalankan.

In [ ]:
# === SALINAN DATA BAWAAN, dipakai hanya bila notebook dijalankan di luar Kaggle ===
# Jangan diubah. Lewati saja cell ini.
_BAWAAN = {}

_BAWAAN['train.csv'] = '''
...
'''

_BAWAAN['test.csv'] = '''
...
'''

_BAWAAN['sample_submission.csv'] = '''
H4sIAMYZamoC/2Xcy66sRw0F4DnPEqGy6z4NSIgRvAHaSBECoYACGfD21BED4q9n51f3/rsu9vLysn1+/9vffP3568e/fPf9
X7/+8/WHf/7w09e//vqPH7/+/qe//fzvrz/+8NP3P//968df/eb737W2WnyXa/569f89R45fPse8p3ze+/zlc+a45bmN8r7s
p77/1udYp/79Gr2+76z691G+H+20Xz731spzmyfL+25m/f3Y5Tn3rM+nPsctz+3kreu7yfcPz2U/cbK+v896PjPr389Zz6vv
et7Zyn7eddTzWLueZ3K/N+vn7xfqfURd7+31/bueRyzOu4963r0d3p+sl/vavd4P9x31vmPPVu+/Jfsf9bl+nu9C8Y96vrPX
3xud99f1xbjsd1V7uO7/9no/1f/aOnU9J+p9ZrXv3Dew9/r9mPW++63rPyPxf9ZX99PGvvhTvb/Vqj2tVdd7An+q/vH2W9e7
wZc563mlfz8G/lR/7/274k2r9tAT/1k8Y3+97rfdas/vgLm/6i/Pvet6x+G8Kj5lH4ffr+ffsec89f2J/cWs78uKvzEbeN0G
/r+wv3o+oxMvKl5lX8QP3rfqejKW8a3uv4Ens4PH9b5y1Pj4zOmCt/V8NvZxPvbLeur+U/uJ6h9tz4qnu+JXzOqfz3/q9wN7
btV+89b48uI/54V99L6J13y/3uc7b/Cp+tvDD/CF/Q3i3Vm+v9rjOb6/xtfesedFvALvNni4sc8OH1pR40108KuuPxf7u8S3
5+AVH+APq8bjh9fGx3r+YzXspRE/6vd7NvwBfjGJl+Dvqutvh3ibF/y82P8G3/i9Niv+HPgG8e6tD/9d9XP5bRBvAnvNVe0j
wO+28T/wt9fzeNfX4L/sj/PaB/6mv3fiZa945d+Per+58fds8Hnw5lb7D+w73gnx9wt/5/yMJ814GNwf9lX5cSbxYVQ+/+wT
/53cV+M8+f1+yScG+EM8hU88fyTfmvCFBT85xMfqby/dm5wP+dIgnyKfafDDCx/m/l94qecxN++v+dCLn8QP+GV88Jl6PgG/
OzX/asT3mOaX8OdR+U2GeFfzneev8M+aXz26Bb5h7x/xpD4/PAYfqv89++3gFfHAfId8Met+n3+Cj/gb+Xi0wf1f/GWxf/SH
OMTfjb1VPG/e1+X7a8N/4APi1SQf2BUfAzxug3x0wQ8n+HvIFw753Krve+4N35zgQ8g3q7+R/738xP3zffjFafC9et/vfXX9
wfld8GbBNxr6CnrQu8CLvx7iexBvwEf1g+q/7/zwJ/M/8tMgP9iVbyb6SsCHvwUE8Aj+wvM+6kvgKfrAlE+QT13i7TR/6eQf
3E9HP9t8nvX+Hx1DH6r49MwF/gI+oue8eOT7GvG63s+s+BMbPWyip6EnJvnjM1D4OPm2emcu1hfYv3xb+wfvVuWTz5/YD+9f
5iPqvTUfefhU7/cYn8jf7tVf4MvEz42+c9nPlP+jd0zsramH8L5d7fHF2+qfiX1t+OyZ6GHuD/6x0FMH/GJgP+OAJxWP8yz4
6Ez0KfTZy/2h3x/X14m3+Lf2MtTz8e+G/pPogQc9ZVR7ePslv8HeDvnQhG908v92iL9V73/uSbwe8uV63udwP+DH2OBlyneJ
79v18ZzkOxf8Bj8m9Ykg3gTnC1958f6C79jbQO+Dr2q/2OOjw+RLxFPuMy/8q8EfdwMPK149Oos+Ct+Nrp4An4A/Ul96/Nb8
CL30Q9+p9tvh6xd/TvS/6PAX9EU/n+qxC70f/4ghPxnkR/Ab6lmNesIOzp/zvN6/6yHe76W+FfBx/Lv6bxvHfIN8svKPSPCu
Ue/Z6gXcL/nWS7e5b/jwgb9d9NehXkD88b4n9boLn1/oBbPGl2+CDfiunm2+QL2Y/CKo3xz0qEG98KLXXfSBhX4w4e/gb6yF
/oi+hb/F2PJH8Ib6wyReNPSjhj1d8CbRX9Cnm/U37XuJt+hFu+pXL/5z/8QH6o+Pn9bz3aGemvA18mP0/Gu+FdhTmn9bP8Z/
0Q8S/WdwvxN7ntN8ifOnftC2fAQ+BD4O+Pod2L/1B/mX/kf9Y0z9faOvwsfUx9CLmvxmoIehf8uPdqjPB3h3qGeR/6Bfkk80
4/8Az4P1kn/n/cjvqHdgjys9X/SoDp91/4E+Tf1rb75PvR59+uWb1DdYr/Wqjh4z4XdZ48GLJ8Q78dZ4B753/Yt4suBf8v0b
1M8P9TjqCYvzsN8pxXPqAdv7XeQr6AsH/eeCf+bf1D+fv1pvb/Bp+zuoZ8BXAvsjvmeqP1tfpH5mPW3Bv2a6f/DL+LPlw8Rj
4t+ing4epvHugpfHegr5R8PfBvWwMF9L6gPUUxf1kEP/FPt78UF/p/6L/X70q8if4acXfjCxN+rpiZ4c1q8v99+11w/9n8/J
R9P8Hnu7xPN70IvQ3w94rj551U/pf4APvXCBPk59aaA/p/oz+LCn+Qj6I/0ba6vPEl+o186tPVI/S/vB5NfwI/XdRF8jP8Ve
Hp+FX5KvNevx3XoP/XzoVwc8DPnbFi8v68dfwffWxVfsx/4Q9WbWa38G+e3DM/qfhvlWo55IvyV4t9D7t/U69foJXqV6lPpv
xz/5e/CU/pxUv44JnqHX0c/x4gH1BPkV+Gv99Ph9+OMd4tEl3yDfZr2N+j36Ywz7i/Hv9tFvQj8T/mY/aKA/0U/9nvVf87GJ
vgQekb8kn9Of28yv+lJ/FE/lu6y3qz/pP+iT6L32N23qo563eOb9LOz3Tuwb/k1+8q2BDn2B+qv2Sn191fga1mev/RbdeiX4
0O3Hof+c/GH6+UK/WvbPgy/016C/JHznpTvme3xffwT/B3pENPUa9KxQ/8ffwvoC/SbqbeA1+dZzX/ub6b+wPki94PB71Pfa
tf+b+vOynwU9KNArWzd+wO/I5zt4Qz7xzgd8n/aTw9e4/2s+Tz48yFcTPm79orGewB4y1NvR57kv8s+g/hMd/vLRX0T8Zb7g
0Wv4APqH/bDoaQ9+0WeIL9RnGvXGb4Ia+rX6t/oWegDx2v6dDR+l/+e5u/bOPAD9Tsf+B85nf+iz3Cf8kPpiUF95dFo9Avvd
zkORj9JPHNP+LvS7a36KHn09X/RN6k25rSfbT+95wl/w38P80L7O8zBvxn47/T0HvZN+38ff6edDzz7U06/1YOo/1/ky+1PC
+p74RT5HfXGE/Ij8Fv04mS8AH5v5RiT1hGb/pf1a8gPul3xqoPc29LbmPF04D0A/BXxavdP5oUR/uPCHpH/uwqfA3xdPzdfZ
D+dHv9C3Aif+Qf671dPxP/yffOj9HPVJ+m/sX6Z/Io/zSfCRI79xPrE7r4c9TO8PvZH6oPWESX47yK+b/SzgwSLeja0/2F+K
/zT1Hudn6PchH7lp/4B8lGfOk36iF9/YH/MM0/4J+MAGb+xXCfRs+lcf/9P+1CfJP6030T+h/vBRb7P+5LwD+WNDrw7nLdE/
h3o+/hDqtR94h17O+pP+ukW/SlDvUa90/ph+32fP6IHOx5FPTPB5LPVJ+wPtJ1Uvp//DeQ74V5Cv2A87WY/zRpf42L3/aT0X
vHU+2Hot+R39bQ/v0Cu0T+eByN/pf2jg0cMv9KnlvAvxmnmkj/oL+Wtir/Z3XfJn5vsa9tCYbw7zAfNB5j/S+i/91s3+mFA/
Ip9I5/GJN2dY32Je+aP/1Pln+bH9DPZfYY9dPQf/pP5Ff30u+pFZT9B/2+h/auqbrdnvCX+g3jecn7M/cdt/S35LPCGeJXw9
4IPfBO36PvVb5x/AM+f/NvEf/vjwD32B+N/kP9t+KvjqsP/N+j/9DOgVS3457C+mv08+g97APGwb4oH+h97ZrF9ob9bjqfeN
a3+b+IHeQX5w4CvMTzb7iy7v4/ya/aO51aN51t6oVxznA/w94oPn2z/8C/s9zvuxX+PXdJ6P81BfaeKv84voP9gD882JPzw+
Zf899VQ+/9DHmTfe5p/D/iHnO+z/OfBZ+kWI/8P/7wJ+81mPNB/mfOEDzOcG+mYwT5/MD+R2Pg9/ML9hXj/lI+BJ2A8z6I/j
/49p9isS3yLRo+Q3H/XYrZ5CPn/sl0TfQ39I+wHoJw37i7f9MPAD65Hb/I/1wPfpTwz6UQL7Sf5/n2b/9QYPlvN/xAP+f49k
/uzZA/oI+ulHfyV8kf6g9P8TYH792Qf5zf/95b/6UIoyJ0sAAA==
'''

print('Salinan bawaan siap:', list(_BAWAAN))

In [ ]:
import os, glob, base64, gzip

def _cari_folder_kaggle():
    for pola in ['/kaggle/input/*/train.csv', '/kaggle/input/train.csv']:
        hit = glob.glob(pola)
        if hit:
            return os.path.dirname(hit[0])
    return None

DATA_DIR = _cari_folder_kaggle()

if DATA_DIR is None and os.path.exists('train.csv'):
    DATA_DIR = '.'

if DATA_DIR is None:
    # Tidak di Kaggle dan berkas belum ada, tulis dari salinan bawaan notebook
    for nama, blob in _BAWAAN.items():
        with open(nama, 'wb') as f:
            f.write(gzip.decompress(base64.b64decode(''.join(blob.split()))))
    DATA_DIR = '.'
    print('Data ditulis dari salinan bawaan notebook.')

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f'Sumber data : {DATA_DIR}')
print(f'train : {train.shape}')
print(f'test  : {test.shape}')
train.head()

## 2. Kenali Datanya

In [ ]:
train.info()
print()
print('Missing value pada train:')
print(train.isna().sum()[lambda s: s > 0].to_string())
print()
print('Missing value pada test:')
print(test.isna().sum()[lambda s: s > 0].to_string())

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(train[TARGET], kde=True, ax=ax[0], color='steelblue')
ax[0].set_title('Distribusi Biaya Operasional')
ax[0].set_xlabel('juta rupiah per bulan')
sns.boxplot(x=train[TARGET], ax=ax[1], color='steelblue')
ax[1].set_title('Boxplot Biaya Operasional')
plt.tight_layout(); plt.show()

print(train[TARGET].describe().round(2).to_string())

In [ ]:
num_cols = train.select_dtypes(include=[np.number]).columns.drop(TARGET)
kor = train[num_cols.tolist() + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False)

plt.figure(figsize=(9, 5))
sns.barplot(x=kor.values, y=kor.index, palette='vlag')
plt.axvline(0, color='black', linewidth=1)
plt.title('Korelasi Fitur Numerik terhadap Biaya Operasional')
plt.show()

kor.round(3).to_frame('korelasi')

> **Perhatikan baik-baik.** Ada beberapa jebakan yang sengaja ditanam di dataset ini, dan semuanya sudah dibahas di lab.
>
> 1. Ada sepasang kolom yang isinya praktis informasi yang sama. Cek VIF-nya.
> 2. Ada kolom yang sama sekali tidak berhubungan dengan target. Menyertakannya tidak menolong.
> 3. Hubungan antara jumlah nasabah dan biaya **tidak sepenuhnya garis lurus**.
> 4. Ada missing value di dua kolom, dan cara Anda mengisinya berpengaruh pada skor.

## 3. Baseline: Selalu Menebak Rata-rata

In [ ]:
X = train.drop(columns=[TARGET, ID])
y = train[TARGET]

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

tebakan = np.full(len(y_val), y_tr.mean())
rmse_baseline = np.sqrt(mean_squared_error(y_val, tebakan))
print(f'RMSE baseline pada data validasi: {rmse_baseline:.3f}')
print('Model apa pun yang Anda buat wajib mengalahkan angka ini.')

## 4. Model Pertama: Linear Regression dengan Pipeline

Kolom kategorikal perlu di-encode dan missing value perlu diisi. Semua itu kita bungkus dalam `Pipeline` supaya **tidak terjadi data leakage**, persis seperti yang dijelaskan pada Bagian 4.5 di lab.

In [ ]:
# Pakai select_dtypes, bukan perbandingan dtype == 'object'. Pada pandas versi 3
# kolom teks bertipe 'str' sehingga pengecekan 'object' akan meleset.
fitur_num = X.select_dtypes(include=[np.number]).columns.tolist()
fitur_kat = [c for c in X.columns if c not in fitur_num]

print('Fitur numerik    :', fitur_num)
print('Fitur kategorikal:', fitur_kat)

prapemrosesan = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), fitur_num),
    ('kat', OneHotEncoder(handle_unknown='ignore', drop='first'), fitur_kat),
])

model_linear = make_pipeline(prapemrosesan, StandardScaler(with_mean=False), Ridge(alpha=1.0))
model_linear.fit(X_tr, y_tr)

rmse_linear = np.sqrt(mean_squared_error(y_val, model_linear.predict(X_val)))
print(f'\nRMSE baseline : {rmse_baseline:.3f}')
print(f'RMSE linear   : {rmse_linear:.3f}')

## 5. Validasi yang Lebih Jujur dengan Cross Validation

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

skor = cross_val_score(model_linear, X, y, cv=kf, scoring='neg_root_mean_squared_error')
print(f'RMSE cross validation : {-skor.mean():.3f} (standar deviasi {skor.std():.3f})')
print()
print('Pakai angka cross validation ini untuk membandingkan model, bukan skor leaderboard.')
print('Leaderboard publik hanya memakai 30 persen data uji sehingga mudah menyesatkan.')

## 6. Membuat Submission Pertama

In [ ]:
def buat_submission(model, nama_berkas='submission.csv'):
    # Latih model pada SELURUH data latih, lalu tulis berkas submission
    model.fit(X, y)
    prediksi = model.predict(test.drop(columns=[ID]))

    sub = pd.DataFrame({ID: test[ID], TARGET: prediksi})
    sub.to_csv(nama_berkas, index=False)

    print(f'Tersimpan: {nama_berkas}  ({len(sub)} baris)')
    print(f'Rentang prediksi: {prediksi.min():.2f} sampai {prediksi.max():.2f} juta')
    return sub


submission = buat_submission(model_linear)
submission.head()

Unggah `submission.csv` ke tab **Submit Predictions** di halaman kompetisi.

Pastikan berkasnya punya tepat dua kolom, yaitu `IDCabang` dan `BiayaOperasional_jutaPerBulan`, dengan 1200 baris.

---

## 7. Giliran Anda Memperbaiki

Di bawah ini beberapa arah perbaikan, diurutkan dari yang paling besar dampaknya. Semuanya memakai teknik yang sudah Anda pelajari.

| Arah | Rujukan di Lab | Perkiraan dampak |
|------|----------------|------------------|
| Tangani multikolinearitas, buang kolom kembar | Bagian 6 | interpretasi jauh lebih rapi |
| Tambahkan suku polinomial derajat 2 | Bagian 8 | **paling besar** |
| Setel `alpha` Ridge atau Lasso lewat cross validation | Bagian 7 | sedang |
| Coba strategi imputasi lain, misalnya per kelompok kota | - | kecil sampai sedang |
| Buang fitur yang tidak signifikan | Bagian 5 | kecil, model jadi sederhana |
| Buat fitur baru, misalnya rasio nasabah per teller | - | bervariasi, layak dicoba |

Satu peringatan penting. Model yang paling rumit belum tentu menang. Tim yang menaikkan derajat polinomial sampai tinggi akan melihat skor latihnya cantik lalu jatuh di leaderboard privat. Itu persis pelajaran overfitting dari Bagian 7.

In [ ]:
# ============================================================
# RUANG KERJA ANDA
# ============================================================
# Contoh kerangka. Ubah, tambah, dan bandingkan sesuka Anda.
#
# model_saya = make_pipeline(
#     prapemrosesan,
#     StandardScaler(with_mean=False),
#     PolynomialFeatures(2, include_bias=False),
#     Ridge(alpha=10.0),
# )
#
# skor = cross_val_score(model_saya, X, y, cv=kf, scoring='neg_root_mean_squared_error')
# print(f'RMSE cross validation: {-skor.mean():.3f}')
#
# Kalau sudah lebih baik dari model linear, buat submission barunya:
# buat_submission(model_saya, 'submission_v2.csv')

---

## Daftar Periksa Sebelum Mengumpulkan

- [ ] Berkas submission punya 1200 baris, di luar baris header.
- [ ] Nama kolomnya tepat `IDCabang` dan `BiayaOperasional_jutaPerBulan`.
- [ ] Tidak ada nilai kosong maupun tak terhingga pada kolom prediksi.
- [ ] Model dilatih pada **seluruh** data latih, bukan hanya potongan latihnya saja.
- [ ] Anda memilih model berdasarkan skor cross validation, bukan skor leaderboard publik.

Selamat berlomba.